At a fintech company a computation pipeline.

Pipeline Scale:

• 5 TB daily input
• 150K events/sec peak
• Join with 18 GB reference table
• 7-day rolling aggregations
• Writes to Delta Lake
• Downstream ML depends on it
• SLA: 40 minutes

Suddenly:

• Runtime increases 5x
• Shuffle spill jumps 12x
• 4 tasks run 30x longer than others
• Executor memory pressure spikes
• No code was deployed

Yesterday: 32 minutes.
Today: 2 hours 47 minutes.

Same code.
Same cluster.
Same dataset.


events_df:
(user_id, event_timestamp, event_date, amount, device_type, country)

user_profile_df:
(user_id, user_segment, risk_score, account_status)

Join key → user_id
Partition key → event_date

🎯 Questions

1 - How do you detect skew programmatically?

2 - How do you inspect physical plan?

3 - Did Spark change join strategy?

4 - How do you enforce deterministic join behavior?

5 - How do you isolate heavy keys?
 
6 - How do you prevent state explosion?

7 - How do you scale this 3× safely?



GENREATE DATA

In [0]:
from pyspark.sql import functions as F

num_users = 30

user_profile_df = spark.range(num_users).select(
    
    F.concat(F.lit("user_"), F.col("id")).alias("user_id"),
    
    F.element_at(
        F.array(
            F.lit("new"),
            F.lit("regular"),
            F.lit("premium"),
            F.lit("vip")
        ),
        (F.floor(F.rand()*4)+1).cast("int")
    ).alias("user_segment"),
    
    F.round(F.rand()*100,2).alias("risk_score"),
    
    F.element_at(
        F.array(
            F.lit("active"),
            F.lit("suspended"),
            F.lit("closed")
        ),
        (F.floor(F.rand()*3)+1).cast("int")
    ).alias("account_status")
)

user_profile_df.show()

In [0]:
from pyspark.sql import functions as F

events_df = spark.range(1000).select(
    
    F.concat(F.lit("user_"), F.floor(F.rand()*30)).alias("user_id"),
    
    F.expr(
        "timestamp_seconds(unix_timestamp(current_timestamp()) - int(rand()*604800))"
    ).alias("event_timestamp"),
    
    F.to_date(
        F.expr(
            "timestamp_seconds(unix_timestamp(current_timestamp()) - int(rand()*604800))"
        )
    ).alias("event_date"),
    
    F.round(F.rand()*500,2).alias("amount"),
    
    F.element_at(
        F.array(
            F.lit("mobile"),
            F.lit("web"),
            F.lit("tablet")
        ),
        (F.floor(F.rand()*3)+1).cast("int")
    ).alias("device_type"),
    
    F.element_at(
        F.array(
            F.lit("US"),
            F.lit("UK"),
            F.lit("India"),
            F.lit("Germany")
        ),
        (F.floor(F.rand()*4) + 1).cast("int")
    ).alias("country")
)

events_df.show()

How do you detect skew programmatically?

In [0]:
events_df.groupBy(F.col('event_date')).count().show()

How do you inspect physical plan?

In [0]:
events_df.groupBy(F.col('event_date')).count().show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false



+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow

      +- PhotonGroupingAgg(keys=[event_date#13849], functions=[finalmerge_count(merge count#13873L) AS count(1)#13871L])


         +- PhotonShuffleExchangeSource

            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#9652]

               +- PhotonShuffleExchangeSink hashpartitioning(event_date#13849, 16)

                  +- PhotonGroupingAgg(keys=[event_date#13849], functions=[partial_count(1) AS count#13873L])

                     +- PhotonProject [cast(timestamp_seconds((1773246756 - cast(cast((rand(3475948758925219789) * 604800.0) as int) as bigint))) as date) AS event_date#13849]
                     
                        +- PhotonRange Range (0, 1000, step=1, splits=8)


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
events_df.join(user_profile_df, on="user_id", how="left").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
   
      +- PhotonProject [user_id#13847, event_timestamp#13848, event_date#13849, amount#13850, device_type#13851, country#13852, user_segment#13738, risk_score#13739, account_status#13740]
         +- PhotonBroadcastHashJoin [user_id#13847], [user_id#13737], LeftOuter, BuildRight, false, true
            :- PhotonProject [concat(user_, cast(FLOOR((rand(9136859053408799885) * 30.0)) as string)) AS user_id#13847, timestamp_seconds((1773247167 - cast(cast((rand(9054110071803283104) * 604800.0) as int) as bigint))) AS event_timestamp#13848, cast(timestamp_seconds((1773247167 - cast(cast((rand(-6827958040538700437) * 604800.0) as int) as bigint))) as date) AS event_date#13849, round((rand(1587641172057029430) * 500.0), 2) AS amount#13850, element_at([mobile,web,tablet], cast((FLOOR((rand(6556929906686414678) * 3.0)) + 1) as int), None, true) AS device_type#13851, element_at([US,UK,India,Germany], cast((FLOOR((rand(2078131230836201578) * 4.0)) + 1) as int), None, true) AS country#13852]
            :  +- PhotonRange Range (0, 1000, step=1, splits=8)
            +- PhotonShuffleExchangeSource
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#9824]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonProject [concat(user_, cast(id#13736L as string)) AS user_id#13737, element_at([new,regular,premium,vip], cast((FLOOR((rand(6967068647035483993) * 4.0)) + 1) as int), None, true) AS user_segment#13738, round((rand(5309772674319972429) * 100.0), 2) AS risk_score#13739, element_at([active,suspended,closed], cast((FLOOR((rand(8541853767677424048) * 3.0)) + 1) as int), None, true) AS account_status#13740]
                        +- PhotonRange Range (0, 30, step=1, splits=8)


== Photon Explanation ==
The query is fully supported by Photon.

Quick Debugging Checklist

When reading any Spark plan, check this order:

1️⃣ Join type

Look at:

PhotonBroadcastHashJoin ... LeftOuter, BuildRight

Questions to ask:

Is the join type correct? (inner, left, broadcast, etc.)

Is Spark broadcasting the right table?

In our plan:

BuildRight
PhotonRange (0,30)

This means:

user_profile_df has 30 rows
,Spark broadcasts it to all executors

Bad sign would be:

SortMergeJoin
because that means big shuffle.

✅ This is correct and efficient

2️⃣ Broadcast vs Shuffle

Look for:

Exchange
, Shuffle

In our plan:

PhotonShuffleExchangeSource
PhotonShuffleMapStage
PhotonShuffleExchangeSink

A shuffle means:

Data moves across executors

Network + disk cost

Check where data originates.

You have:

PhotonRange Range (0, 1000)
PhotonRange Range (0, 30)

This means:

Data is synthetically generated

No file scans (Parquet/Delta)

Normally you'd see:

FileScan parquet  
DeltaScan

which tells you how much data Spark reads.

3️⃣ Number of shuffles


4️⃣ Partition counts
Look for:

splits=8

This means:

Range (0,1000, splits=8)

Spark created 8 partitions.

Ask:

Too few partitions → slow parallelism

Too many partitions → scheduler overhead

5️⃣ Scan size

6️⃣ Expensive operations (UDFs)

7️⃣ Photon / WholeStageCodegen
You see:

AdaptiveSparkPlan isFinalPlan=false

This means:

Spark will optimize during runtime, e.g.:

convert shuffle join → broadcast join

change partition counts

This is good.

You also see:

PhotonResultStage.                            
PhotonProject                                 
PhotonRange

and:

The query is fully supported by Photon.

Meaning:

query runs on Photon engine

uses vectorized C++ execution

🚀 Faster than standard Spark.



**Spark Skew Join optimization**

Spark can automatically split skewed partitions.

`spark.sql.adaptive.skewJoin.enabled = true`

**How do you prevent state explosion?**

`groupBy(window(event_time,"5 minutes"), user_id)` is mainly used in Spark Structured Streaming to create time-bounded aggregations.
It helps with **incremental processing, state cleanup, and scalability**.


The window() function groups events into fixed time buckets.

Streaming systems continuously receive data.

**Without windows:**

`groupBy(user_id).sum(amount)`

Spark must maintain state forever for every user.

**Example:**

`user1 → running sum
user2 → running sum
user3 → running sum
...`

**With windows:**

`groupBy(window(event_time,"5 minutes"), user_id)`
Spark only keeps state for active windows.

Example state store:
`
10:00-10:05 → user1, user2
10:05-10:10 → user1
`
Once a window is complete, Spark can drop it.

This prevents state explosion.


**Combine with Watermark**

Usually used like this:

`events_df \
.withWatermark("event_time","10 minutes") \
.groupBy(
    window("event_time","5 minutes"),
    "user_id"
) \
.sum("amount")
`
Meaning:

wait 10 minutes for late data, then finalize window

remove state

**Example:**

window: 10:00-10:05

watermark: 10:15

After 10:15 → Spark deletes window state.